# NCTBench-Pilot — Step 6: AMSV Validation
6-signal automated validation. Core methodological contribution.

Pipeline flow:
Raw QA → S3 Extractability → S4 Language → S5 Structure
       → S2 LLM Verify → S1 Retrieval → S6 Cross-Model
       → Dedup → Validated QA → Gold Test Set

In [ ]:
import sys, json, matplotlib.pyplot as plt
from pathlib import Path
sys.path.insert(0, str(Path(r"E:\CSAA Project\pipeline")))
from config import VALIDATED_DIR, CMAR_TARGET

## Individual Signal Demonstrations

In [ ]:
from step6_amsv_validate import signal3_overlap
good = {
    "answer": "RAM stores temporary data for fast processor access",
    "source_text": (
        "RAM or Random Access Memory stores temporary data "
        "for the processor to access quickly during computation"
    ),
}
bad = {
    "answer": "The moon is made of cheese and rocks",
    "source_text": "RAM stores temporary data for the processor",
}
ok1, s1 = signal3_overlap(good)
ok2, s2 = signal3_overlap(bad)
print(f"Signal 3 — Answer Extractability")
print(f"  Good pair: passed={ok1}  overlap={s1:.3f}")
print(f"  Bad pair : passed={ok2}  overlap={s2:.3f}")

In [ ]:
from step6_amsv_validate import signal4_language, signal5_structure
test_s4 = [
    {"question": "What is RAM?", "language": "en"},
    {"question": "RAM কী?", "language": "bn"},
    {"question": "RAM কী?", "language": "en"},
]
print("Signal 4 — Language Integrity")
for p in test_s4:
    passed, detected = signal4_language(p)
    print(f"  Q: {p['question']:20s} "
          f"declared={p['language']} "
          f"detected={detected} passed={passed}")

test_s5 = [
    {"question": "What is the main function of RAM?",
     "answer": "RAM stores data for fast processor access.",
     "evidence": "RAM stores temporary data"},
    {"question": "What?", "answer": "Yes.", "evidence": ""},
]
print("\nSignal 5 — Structural Quality")
for p in test_s5:
    passed, reason = signal5_structure(p)
    print(f"  Q: {p['question']:40s} passed={passed} ({reason})")

## Run Full AMSV Pipeline

In [ ]:
from step6_amsv_validate import run_amsv
run_amsv()

In [ ]:
report = json.loads(
    (VALIDATED_DIR / "validation_report.json")
    .read_text(encoding="utf-8")
)
print("AMSV VALIDATION REPORT")
print("=" * 55)
rows = [
    ("Raw pairs generated",     "total_raw"),
    ("After S3 + S4 + S5",      "after_s3_s4_s5"),
    ("After S2 (LLM verify)",   "after_s2"),
    ("After S1 (retrieval)",    "after_s1"),
    ("After S6 (cross-model)",  "after_s6"),
    ("After deduplication",     "after_dedup"),
    ("Gold test set",           "gold_test_set_size"),
]
for label, key in rows:
    val = report.get(key, 0)
    pct = val / max(report["total_raw"], 1) * 100
    print(f"  {label:30s}: {val:5d}  ({pct:5.1f}%)")
print()
print(f"  CMAR           : {report['cmar']:.4f}")
print(f"  CMAR target    : >= {report['cmar_target']}")
print(f"  CMAR met       : {report['cmar_met']}")
print(f"  Rejection rate : {report['rejection_rate']*100:.1f}%")

In [ ]:
stages = [
    "Raw", "After S3+S4+S5", "After S2 LLM",
    "After S1 Retrieval", "After S6 Cross-Model",
    "After Dedup", "Gold Test Set"
]
counts = [
    report["total_raw"], report["after_s3_s4_s5"],
    report["after_s2"], report["after_s1"],
    report["after_s6"], report["after_dedup"],
    report["gold_test_set_size"]
]
colors = ["#0072B2"] * 6 + ["#009E73"]
fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(stages, counts, color=colors)
for bar, count in zip(bars, counts):
    ax.text(bar.get_width() + max(counts) * 0.01,
            bar.get_y() + bar.get_height() / 2,
            str(count), va="center", fontsize=10)
ax.set_xlabel("Number of QA Pairs")
ax.set_title("AMSV Validation Pipeline Funnel", fontsize=13)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.savefig(r"E:\CSAA Project\notebooks\amsv_funnel.png",
            dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
test_set = json.loads(
    (VALIDATED_DIR / "test_set.json")
    .read_text(encoding="utf-8")
)
print(f"Gold test set: {len(test_set)} pairs\n")
for i, p in enumerate(test_set[:3]):
    print(f"--- Pair {i+1} "
          f"[{p['language']} | class{p['class_num']} | "
          f"{p['subject_code']}] ---")
    print(f"Q: {p['question']}")
    print(f"A: {p['answer']}")
    print(f"E: {p.get('evidence','')[:150]}")
    print()